In [1]:
# %%
# ==============================================================================
# DATA PIPELINE CELL 1 – Imports & Global Config
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# ── Reproducibility ────────────────────────────────────────────────────────────
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Paths ──────────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
DATA_DIR     = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
CONFIG_PATH  = PROJECT_ROOT / 'schema_config.json'

DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Sensor schema ──────────────────────────────────────────────────────────────
with open(CONFIG_PATH) as f:
    GRUP_SENSOR = json.load(f)

FITUR_AKTIF = [ft for sub in GRUP_SENSOR.values() for ft in sub]
assert len(FITUR_AKTIF) == len(set(FITUR_AKTIF)), "Duplicate feature names in schema!"
print(f"[OK] Schema loaded: {len(FITUR_AKTIF)} features across {len(GRUP_SENSOR)} groups")

# ── RUL caps (OEM-based) ───────────────────────────────────────────────────────
RUL_CAP_CONFIG = {
    "hydraulic_pump": 3000,
    "main_bearing":   4000,
    "turbocharger":   2500,
    "transmission":   5000,
    "brake_caliper":  1200,
    "brake_pad":       800,
    "pump_seal":      1500,
    "default_wear":   5000,
}
DEFAULT_RUL_CAP = RUL_CAP_CONFIG["default_wear"]

# ── Label thresholds (SINGLE DEFINITION — dipakai semua pipeline) ──────────────
WARNING_THRESH  = 72    # jam
CRITICAL_THRESH = 24    # jam

def create_labels(rul_array: np.ndarray) -> np.ndarray:
    """0=NORMAL, 1=WARNING (<72h), 2=CRITICAL (<24h)"""
    labels = np.zeros_like(rul_array, dtype=np.int32)
    labels[(rul_array >= CRITICAL_THRESH) & (rul_array < WARNING_THRESH)] = 1
    labels[rul_array < CRITICAL_THRESH] = 2
    return labels

LABEL_NAMES = {0: 'NORMAL', 1: 'WARNING', 2: 'CRITICAL'}

# ── Sequence length ────────────────────────────────────────────────────────────
SEQUENCE_LENGTH = 24

print(f"[OK] RUL cap: {DEFAULT_RUL_CAP}h | WARNING<{WARNING_THRESH}h | CRITICAL<{CRITICAL_THRESH}h")
print(f"[OK] Sequence length: {SEQUENCE_LENGTH} timesteps")

[OK] Schema loaded: 32 features across 4 groups
[OK] RUL cap: 5000h | WARNING<72h | CRITICAL<24h
[OK] Sequence length: 24 timesteps


In [2]:
# %%
# ==============================================================================
# DATA PIPELINE CELL 2 – Synthetic Fleet Generation (30 units, physics-compliant)
# ==============================================================================
N_UNITS = 30

type_specs = {
    'haul_truck': {
        'count': 12, 'names': [f"HD785-{i+1:03d}" for i in range(12)],
        'road_grade_mean': 6.0, 'road_grade_std': 2.5, 'road_grade_min': 0.5,
        'payload_mean': 165, 'payload_std': 22, 'payload_min': 80, 'payload_max': 220,
        'engine_rpm_mean': 1800, 'engine_rpm_std': 30,
        'engine_load_mean': 65, 'engine_load_std': 12,
        'hydraulic_pressure_nom': 280, 'cycle_time_factor': 25.0,
        'hours_min': 2000, 'hours_max': 8000, 'has_brake_hierarchy': True,
    },
    'excavator': {
        'count': 6, 'names': [f"PC2000-{i+1:03d}" for i in range(6)],
        'road_grade_mean': 0.0, 'road_grade_std': 0.5, 'road_grade_min': 0.0,
        'payload_mean': 180, 'payload_std': 20, 'payload_min': 120, 'payload_max': 240,
        'engine_rpm_mean': 1900, 'engine_rpm_std': 20,
        'engine_load_mean': 70, 'engine_load_std': 10,
        'hydraulic_pressure_nom': 320, 'cycle_time_factor': None,
        'hours_min': 3000, 'hours_max': 8000, 'has_brake_hierarchy': False,
    },
    'bulldozer': {
        'count': 6, 'names': [f"D155-{i+1:03d}" for i in range(6)],
        'road_grade_mean': 1.0, 'road_grade_std': 1.0, 'road_grade_min': 0.0,
        'payload_mean': 0, 'payload_std': 0, 'payload_min': 0, 'payload_max': 0,
        'engine_rpm_mean': 1700, 'engine_rpm_std': 25,
        'engine_load_mean': 80, 'engine_load_std': 12,
        'hydraulic_pressure_nom': 300, 'cycle_time_factor': None,
        'hours_min': 2500, 'hours_max': 7500, 'has_brake_hierarchy': False,
    },
    'wheel_loader': {
        'count': 6, 'names': [f"WA600-{i+1:03d}" for i in range(6)],
        'road_grade_mean': 2.0, 'road_grade_std': 1.0, 'road_grade_min': 0.0,
        'payload_mean': 150, 'payload_std': 15, 'payload_min': 100, 'payload_max': 180,
        'engine_rpm_mean': 1750, 'engine_rpm_std': 28,
        'engine_load_mean': 75, 'engine_load_std': 10,
        'hydraulic_pressure_nom': 290, 'cycle_time_factor': None,
        'hours_min': 2000, 'hours_max': 7000, 'has_brake_hierarchy': False,
    },
}

# ── Build base dataframe ───────────────────────────────────────────────────────
unit_data = []
for typ, spec in type_specs.items():
    for name in spec['names']:
        hrs   = int(rng.integers(spec['hours_min'], spec['hours_max'] + 1))
        start = pd.Timestamp("2024-01-01") + pd.Timedelta(hours=int(rng.integers(0, 8760)))
        df_unit = pd.DataFrame({
            "timestamp":      pd.date_range(start=start, periods=hrs, freq="h"),
            "asset_id":       name,
            "equipment_type": typ,
            "RUL_hours":      np.arange(hrs - 1, -1, -1, dtype=np.float32),
        })
        unit_data.append(df_unit)

df = (pd.concat(unit_data, ignore_index=True)
        .sort_values(["asset_id", "timestamp"])
        .reset_index(drop=True))
N = len(df)
print(f"[INFO] {N:,} rows | {df['asset_id'].nunique()} units | {df['equipment_type'].nunique()} types")

# ── Derived scalars ────────────────────────────────────────────────────────────
asset_spec_map = {name: spec for typ, spec in type_specs.items() for name in spec['names']}
def _spec(asset, key): return asset_spec_map[asset][key]

max_rul          = df.groupby('asset_id')['RUL_hours'].transform('max')
hi               = np.clip(df['RUL_hours'] / max_rul, 0.0, 1.0).values
degradation_factor = np.exp(-5.0 * hi)
MAX_OIL_INTERVAL   = 500.0
hours_since_oil    = ((1.0 - hi) * MAX_OIL_INTERVAL).clip(0, MAX_OIL_INTERVAL - 10)
truck_mask         = (df['equipment_type'] == 'haul_truck').values

# ── Environmental context ──────────────────────────────────────────────────────
rg_mean = df['asset_id'].map(lambda a: _spec(a, 'road_grade_mean')).values
rg_std  = df['asset_id'].map(lambda a: _spec(a, 'road_grade_std')).values
pl_mean = df['asset_id'].map(lambda a: _spec(a, 'payload_mean')).values
pl_std  = df['asset_id'].map(lambda a: _spec(a, 'payload_std')).values

df['road_grade_pct']       = np.abs(rng.normal(rg_mean, rg_std, N)).clip(0, 12).astype(np.float32)
df.loc[truck_mask, 'road_grade_pct'] = df.loc[truck_mask, 'road_grade_pct'].clip(0.5, 12.0)
df['payload_tonnage']      = rng.normal(pl_mean, pl_std, N).clip(0, 240).astype(np.float32)
df.loc[df['equipment_type'] == 'bulldozer', 'payload_tonnage'] = 0.0
df['haul_distance_km']     = np.where(truck_mask, rng.normal(3.5, 0.8, N).clip(0.5, 8.0), 0.0).astype(np.float32)
df['cycle_time_minutes']   = np.where(truck_mask, (df['haul_distance_km'] / 25.0 * 60 + rng.normal(0, 3, N)).clip(5, 60), 0.0).astype(np.float32)
df['dust_concentration_mgm3'] = (8.0 + df['road_grade_pct'] * 1.2 + rng.normal(0, 2, N)).clip(1, 50).astype(np.float32)
df['humidity_pct']         = rng.normal(78, 10, N).clip(40, 98).astype(np.float32)
df['ambient_temp_c']       = rng.normal(32, 4, N).clip(22, 45).astype(np.float32)
df['days_since_last_pm']   = ((1.0 - hi) * 90.0 + rng.uniform(0, 5, N)).clip(0, 180).astype(np.float32)

# ── Engine telemetry ───────────────────────────────────────────────────────────
rpm_mean  = df['asset_id'].map(lambda a: _spec(a, 'engine_rpm_mean')).values
rpm_std   = df['asset_id'].map(lambda a: _spec(a, 'engine_rpm_std')).values
load_mean = df['asset_id'].map(lambda a: _spec(a, 'engine_load_mean')).values
load_std  = df['asset_id'].map(lambda a: _spec(a, 'engine_load_std')).values

df['engine_rpm']               = (rng.normal(rpm_mean, rpm_std, N) - degradation_factor * 40).clip(1400, 2200).astype(np.float32)
df['engine_load_pct']          = rng.normal(load_mean, load_std, N).clip(20, 100).astype(np.float32)
df['exhaust_gas_temp_c']       = (200 + df['engine_load_pct'] * 4.5 + df['ambient_temp_c'] * 1.2 + degradation_factor * 80 + rng.normal(0, 5, N)).clip(180, 750).astype(np.float32)
df['engine_oil_temp_c']        = (90 + degradation_factor * 35 + rng.normal(0, 2, N)).clip(75, 140).astype(np.float32)
df['transmission_oil_temp_c']  = (85 + degradation_factor * 28 + rng.normal(0, 2, N)).clip(70, 130).astype(np.float32)
df['coolant_temp_c']           = (88 + degradation_factor * 20 + rng.normal(0, 2, N)).clip(75, 120).astype(np.float32)
df['coolant_pressure_kpa']     = (180 - degradation_factor * 25 + rng.normal(0, 3, N)).clip(100, 220).astype(np.float32)
df['engine_oil_pressure_kpa']  = (420 - degradation_factor * 60 + rng.normal(0, 5, N)).clip(200, 500).astype(np.float32)
df['transmission_oil_pressure_kpa'] = (350 - degradation_factor * 45 + rng.normal(0, 4, N)).clip(150, 420).astype(np.float32)
df['fuel_consumption_rate_lph'] = (80 + df['engine_load_pct'] * 0.8 + degradation_factor * 15 + rng.normal(0, 3, N)).clip(40, 180).astype(np.float32)
df['boost_pressure_kpa']       = (220 - degradation_factor * 30 + rng.normal(0, 4, N)).clip(100, 280).astype(np.float32)
df['battery_voltage_v']        = rng.normal(24.5, 0.3, N).clip(22, 27).astype(np.float32)

# ── Hydraulic ──────────────────────────────────────────────────────────────────
pnom     = df['asset_id'].map(lambda a: _spec(a, 'hydraulic_pressure_nom')).values
hyd_drop = np.where(hi < 0.30, (0.30 - hi) / 0.30 * 80, 0.0)
df['hydraulic_main_pump_pressure_bar'] = (pnom - hyd_drop + rng.normal(0, 4, N)).clip(100, 350).astype(np.float32)

# ── Vibration (4-stage bearing degradation) ────────────────────────────────────
vib_z = np.zeros(N)
s1 = hi >= 0.70;              vib_z[s1] = 0.80 + rng.normal(0, 0.04, s1.sum())
s2 = (hi >= 0.40) & (hi < 0.70); vib_z[s2] = 0.80 + (0.70 - hi[s2]) / 0.30 * 0.60 + rng.normal(0, 0.07, s2.sum())
s3 = (hi >= 0.15) & (hi < 0.40); vib_z[s3] = 1.40 + (0.40 - hi[s3]) / 0.25 * 1.80 + rng.normal(0, 0.12, s3.sum())
s4 = hi < 0.15;               vib_z[s4] = 3.20 + ((0.15 - hi[s4]) / 0.15) ** 2.5 * 2.80 + rng.normal(0, 0.22, s4.sum())
df['vibration_z_g'] = vib_z.clip(0.2, 8.0).astype(np.float32)
df['vibration_x_g'] = (vib_z * 0.65 + rng.normal(0, 0.05, N)).clip(0.2, 8.0).astype(np.float32)
df['vibration_y_g'] = (vib_z * 0.55 + rng.normal(0, 0.05, N)).clip(0.2, 8.0).astype(np.float32)

# ── Acoustic emission (Paris Law) ─────────────────────────────────────────────
ae = np.zeros(N)
ae[hi >= 0.15] = 45 + (1 - hi[hi >= 0.15]) * 20 + rng.normal(0, 1, (hi >= 0.15).sum())
ae[hi < 0.15]  = 65 + ((0.15 - hi[hi < 0.15]) / 0.15) ** 2.5 * 30 + rng.normal(0, 2, (hi < 0.15).sum())
df['acoustic_emission_db'] = ae.clip(40, 100).astype(np.float32)

# ── Fluid quality ──────────────────────────────────────────────────────────────
df['oil_viscosity_cst']       = (14.5 + 0.008 * hours_since_oil + rng.normal(0, 0.12, N)).clip(13, 19.5).astype(np.float32)
df['oil_particle_count_iso']  = (15 + (hours_since_oil / 100) ** 1.5 + rng.normal(0, 0.25, N)).clip(14, 21).astype(np.float32)
df['oil_moisture_pct']        = (0.02 + df['humidity_pct'] / 100 * 0.08 + hours_since_oil / MAX_OIL_INTERVAL * 0.05 + rng.normal(0, 0.005, N)).clip(0.01, 0.20).astype(np.float32)
df['wear_metal_fe_ppm']       = (5 + degradation_factor * 60 + rng.normal(0, 2, N)).clip(0, 150).astype(np.float32)
df['wear_metal_cu_ppm']       = (2 + degradation_factor * 25 + rng.normal(0, 1, N)).clip(0, 80).astype(np.float32)

# ── Maintenance logs ───────────────────────────────────────────────────────────
df['last_maintenance_hours'] = (df['RUL_hours'] + rng.uniform(50, 500, N)).astype(np.float32)
df['oil_change_flag']        = (hours_since_oil < 10).astype(np.int8)

# ── Brake system (haul truck only) ────────────────────────────────────────────
has_hier = df['asset_id'].map(lambda a: _spec(a, 'has_brake_hierarchy')).values
df['braking_work_proxy'] = np.where(
    truck_mask,
    df['payload_tonnage'] * df['road_grade_pct'] * df['haul_distance_km'],
    0.0
)
df = df.sort_values(['asset_id', 'timestamp']).reset_index(drop=True)
df['cumulative_braking_work'] = df.groupby('asset_id')['braking_work_proxy'].cumsum()
max_bw = df.groupby('asset_id')['cumulative_braking_work'].transform('max')

df['RUL_brake_pad_rear_axle'] = np.where(
    has_hier & (max_bw > 0),
    (RUL_CAP_CONFIG["brake_pad"] * (1 - df['cumulative_braking_work'] / max_bw) + rng.normal(0, 8, N)).clip(0, RUL_CAP_CONFIG["brake_pad"]),
    RUL_CAP_CONFIG["brake_pad"]
).astype(np.float32)

# ── Three-level RUL hierarchy ──────────────────────────────────────────────────
df['RUL_hydraulic_system'] = (df['RUL_hours'] * rng.uniform(1.05, 1.20, N)).clip(0, DEFAULT_RUL_CAP).astype(np.float32)
df['RUL_powertrain']       = (df['RUL_hours'] * rng.uniform(1.05, 1.25, N)).clip(0, DEFAULT_RUL_CAP).astype(np.float32)
df['RUL_brake_caliper']    = np.where(
    has_hier,
    (df['RUL_brake_pad_rear_axle'] * rng.uniform(1.05, 1.15, N)).clip(0, RUL_CAP_CONFIG["brake_caliper"]),
    df['RUL_brake_pad_rear_axle']
).astype(np.float32)
df['RUL_brake_system']     = np.where(
    has_hier,
    (df['RUL_brake_caliper'] * rng.uniform(1.10, 1.20, N)).clip(0, DEFAULT_RUL_CAP),
    df['RUL_brake_caliper']
).astype(np.float32)
df['RUL_steering_system']  = (df['RUL_hours'] * rng.uniform(1.05, 1.15, N)).clip(0, DEFAULT_RUL_CAP).astype(np.float32)
df['RUL_suspension']       = (df['RUL_hours'] * rng.uniform(1.02, 1.12, N)).clip(0, DEFAULT_RUL_CAP).astype(np.float32)

df['RUL_hydraulic_pump']   = (df['RUL_hydraulic_system'] * rng.uniform(0.80, 0.98, N)).clip(0, RUL_CAP_CONFIG["hydraulic_pump"]).astype(np.float32)
df['RUL_main_bearing']     = (df['RUL_powertrain']       * rng.uniform(0.75, 0.95, N)).clip(0, RUL_CAP_CONFIG["main_bearing"]).astype(np.float32)
df['RUL_turbocharger']     = (df['RUL_powertrain']       * rng.uniform(0.70, 0.90, N)).clip(0, RUL_CAP_CONFIG["turbocharger"]).astype(np.float32)
df['RUL_transmission']     = (df['RUL_powertrain']       * rng.uniform(0.85, 0.99, N)).clip(0, RUL_CAP_CONFIG["transmission"]).astype(np.float32)

df['RUL_pump_seal_main']     = (df['RUL_hydraulic_pump'] * rng.uniform(0.70, 0.95, N)).clip(0, RUL_CAP_CONFIG["pump_seal"]).astype(np.float32)
df['RUL_bearing_inner_race'] = (df['RUL_main_bearing']   * rng.uniform(0.65, 0.90, N)).clip(0, RUL_CAP_CONFIG["main_bearing"]).astype(np.float32)
df['RUL_turbo_impeller']     = (df['RUL_turbocharger']   * rng.uniform(0.60, 0.88, N)).clip(0, RUL_CAP_CONFIG["turbocharger"]).astype(np.float32)
df['RUL_brake_pad_rear']     = df['RUL_brake_pad_rear_axle'].copy()

# ── Hierarchy validation ───────────────────────────────────────────────────────
checks = [
    ("RUL_pump_seal_main",     "RUL_hydraulic_pump",  "RUL_hydraulic_system"),
    ("RUL_bearing_inner_race", "RUL_main_bearing",    "RUL_powertrain"),
    ("RUL_turbo_impeller",     "RUL_turbocharger",    "RUL_powertrain"),
    ("RUL_brake_pad_rear",     "RUL_brake_caliper",   "RUL_brake_system"),
]
all_ok = True
for part, comp, subs in checks:
    v1 = (df[part] > df[comp]).sum()
    v2 = (df[comp] > df[subs]).sum()
    if v1 or v2:
        print(f"[HIERARCHY FAIL] {part}: part>comp={v1}, comp>subs={v2}")
        all_ok = False
if all_ok:
    print("[PASS] RUL hierarchy satisfied.")

# ── Clean up & export ──────────────────────────────────────────────────────────
df.drop(columns=['braking_work_proxy', 'cumulative_braking_work'], inplace=True, errors='ignore')
float64_cols = df.select_dtypes('float64').columns
df[float64_cols] = df[float64_cols].astype(np.float32)

out_path = DATA_DIR / "dataset_pratyaksa_noisy.parquet"
df.to_parquet(out_path, index=False)
print(f"[OK] Exported → {out_path} | Shape: {df.shape} | Memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB")

[INFO] 148,198 rows | 30 units | 4 types
[PASS] RUL hierarchy satisfied.
[OK] Exported → /home/sleepy/projects/project-pratyaksa-kic/data/dataset_pratyaksa_noisy.parquet | Shape: (148198, 51) | Memory: 48.9 MB


In [3]:
# %%
# ==============================================================================
# DATA PIPELINE CELL 3 – Sensor Quality Validation
# ==============================================================================
FLATLINE_WIN  = 10
FLATLINE_THR  = 1e-6
GAP_THRESH_MIN = 120

input_cols = [c for c in FITUR_AKTIF if c in df.columns]
df_work = df.copy()

# ── Flatline detection ─────────────────────────────────────────────────────────
print("[FLATLINE DETECTION]")
dropout_cols = []
for col in input_cols:
    mask = df_work.groupby('asset_id')[col].transform(
        lambda s: s.rolling(FLATLINE_WIN, min_periods=FLATLINE_WIN).std() < FLATLINE_THR
    )
    n = int(mask.sum())
    if n > 0:
        print(f"  {col}: {n} flatlines flagged")
        flag_col = f'{col}_dropout_flag'
        df_work[flag_col] = mask.astype(np.int8)
        df_work.loc[mask, col] = np.nan
        dropout_cols.append(flag_col)

if not dropout_cols:
    print("  No flatlines detected.")

# ── Gap detection ──────────────────────────────────────────────────────────────
def flag_gaps(grp):
    grp = grp.sort_values('timestamp')
    gap_minutes = grp['timestamp'].diff().dt.total_seconds().fillna(0) / 60
    return (gap_minutes > GAP_THRESH_MIN).astype(np.int8)

df_work['connectivity_loss_flag'] = (
    df_work.groupby('asset_id', group_keys=False).apply(flag_gaps)
)
total_gaps = int(df_work['connectivity_loss_flag'].sum())
print(f"[GAPS] {total_gaps:,} connectivity loss events flagged.")

# ── NaN imputation (Pandas 3.0.3 Optimized) ────────────────────────────────────
nan_total = df_work[input_cols].isna().sum().sum()
if nan_total > 0:
    print(f"[NaN] {nan_total:,} NaN values — imputing with per-asset forward fill then global mean")
    
    # 1. Forward-fill within groups (C-optimized)
    df_work[input_cols] = df_work.groupby('asset_id')[input_cols].ffill()
    
    # 2. Backward-fill within groups (C-optimized, handles NaNs at the start of an asset's timeline)
    df_work[input_cols] = df_work.groupby('asset_id')[input_cols].bfill()
    
    # 3. Global mean fill (if any whole assets are completely NaN for a feature)
    for col in input_cols:
        if df_work[col].isna().any():
            global_mean = df_work[col].mean()
            df_work[col] = df_work[col].fillna(global_mean)
            
    print("[OK] NaN imputation complete.")
else:
    print("[PASS] No NaN values found.")

# ── Correlation with RUL ───────────────────────────────────────────────────────
corr = df_work[input_cols + ['RUL_hours']].corr()['RUL_hours'].drop('RUL_hours')
print("\n[TOP 10 CORRELATIONS with RUL_hours]")
for feat, val in corr.abs().sort_values(ascending=False).head(10).items():
    direction = '-' if corr[feat] < 0 else '+'
    print(f"  {feat:<35}: {val:.3f} ({direction})")

print(f"\n[SUMMARY] {df_work.shape[0]:,} rows | {len(input_cols)} features | "
      f"{df_work[input_cols].isna().sum().sum()} NaN remaining")

df = df_work.copy()

[FLATLINE DETECTION]
  oil_particle_count_iso: 46011 flatlines flagged
  payload_tonnage: 27518 flatlines flagged
  cycle_time_minutes: 91157 flatlines flagged
  haul_distance_km: 91157 flatlines flagged
  oil_change_flag: 147658 flatlines flagged
[GAPS] 0 connectivity loss events flagged.
[NaN] 403,501 NaN values — imputing with per-asset forward fill then global mean
[OK] NaN imputation complete.

[TOP 10 CORRELATIONS with RUL_hours]
  last_maintenance_hours             : 0.997 (+)
  days_since_last_pm                 : 0.898 (-)
  oil_viscosity_cst                  : 0.895 (-)
  oil_particle_count_iso             : 0.868 (-)
  acoustic_emission_db               : 0.798 (-)
  vibration_z_g                      : 0.791 (-)
  vibration_x_g                      : 0.790 (-)
  vibration_y_g                      : 0.789 (-)
  wear_metal_fe_ppm                  : 0.765 (-)
  wear_metal_cu_ppm                  : 0.762 (-)

[SUMMARY] 148,198 rows | 32 features | 0 NaN remaining


In [4]:
# %%
# ==============================================================================
# DATA PIPELINE CELL 4 – Censoring, Split & Scaling
# ==============================================================================
# ── RUL cap ────────────────────────────────────────────────────────────────────
if df['RUL_hours'].max() > DEFAULT_RUL_CAP:
    print(f"[ACTION] Capping RUL_hours at {DEFAULT_RUL_CAP}h")
    df['RUL_hours'] = df['RUL_hours'].clip(upper=DEFAULT_RUL_CAP)

# ── RIGHT-CENSORING YANG BENAR ─────────────────────────────────────────────────
# Pertahankan SEMUA unit (censored dan failed)
# Tandai mana yang observed failure vs censored
FAILURE_THRESHOLD = 5.0
unit_min_rul = df.groupby('asset_id')['RUL_hours'].min()
failed_units   = unit_min_rul[unit_min_rul <= FAILURE_THRESHOLD].index
censored_units = unit_min_rul[unit_min_rul >  FAILURE_THRESHOLD].index

df['is_failed']       = df['asset_id'].isin(failed_units).astype(np.int8)
df['event_observed']  = df['is_failed']  # 1=failure, 0=right-censored

print(f"[CENSORING] Failed (observed): {len(failed_units)} units | "
      f"Censored (right): {len(censored_units)} units | "
      f"Total retained: {df['asset_id'].nunique()} units")

# ── Time-based split (80/10/10) ────────────────────────────────────────────────
global_sorted = df.sort_values(['timestamp', 'asset_id']).reset_index(drop=True)
n = len(global_sorted)
cutoff_train = global_sorted.loc[int(n * 0.80), 'timestamp']
cutoff_val   = global_sorted.loc[int(n * 0.90), 'timestamp']

train_df = global_sorted[global_sorted['timestamp'] <= cutoff_train].copy()
val_df   = global_sorted[(global_sorted['timestamp'] > cutoff_train) &
                          (global_sorted['timestamp'] <= cutoff_val)].copy()
test_df  = global_sorted[global_sorted['timestamp'] > cutoff_val].copy()

print(f"[SPLIT] Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"        Cutoff train: {cutoff_train} | Cutoff val: {cutoff_val}")

# ── Feature matrix ─────────────────────────────────────────────────────────────
dropout_flag_cols = [c for c in df.columns if c.endswith('_dropout_flag')]
FITUR_FINAL = [c for c in FITUR_AKTIF if c in df.columns] + dropout_flag_cols

X_train_raw = train_df[FITUR_FINAL].values.astype(np.float32)
X_val_raw   = val_df[FITUR_FINAL].values.astype(np.float32)
X_test_raw  = test_df[FITUR_FINAL].values.astype(np.float32)

y_train = train_df['RUL_hours'].values.astype(np.float32)
y_val   = val_df['RUL_hours'].values.astype(np.float32)
y_test  = test_df['RUL_hours'].values.astype(np.float32)

# ── SINGLE SCALER — fit on FULL train set ──────────────────────────────────────
# CRITICAL FIX: scaler di-fit sekali pada train_df penuh
# Tidak ada scaler kedua di pipeline manapun
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled   = scaler.transform(X_val_raw)
X_test_scaled  = scaler.transform(X_test_raw)

joblib.dump(scaler, ARTIFACTS_DIR / "artifact_scaler.pkl")
print(f"[OK] Scaler fitted on {len(X_train_scaled):,} train samples → saved")

# ── Sample weights (failed units weighted 2x) ──────────────────────────────────
train_weights = np.where(train_df['is_failed'] == 1, 1.0, 0.5).astype(np.float32)

# ── Label distribution ─────────────────────────────────────────────────────────
train_labels = create_labels(y_train)
print("\n[LABEL DISTRIBUTION - Train]")
for lbl, name in LABEL_NAMES.items():
    cnt = (train_labels == lbl).sum()
    print(f"  {name:<10}: {cnt:>8,} ({cnt/len(train_labels)*100:.1f}%)")

# ── Simpan ke artifacts ────────────────────────────────────────────────────────
train_df.to_parquet(ARTIFACTS_DIR / "split_train.parquet", index=False)
val_df.to_parquet(ARTIFACTS_DIR  / "split_val.parquet",   index=False)
test_df.to_parquet(ARTIFACTS_DIR / "split_test.parquet",  index=False)

split_meta = {
    "train_assets": train_df['asset_id'].unique().tolist(),
    "val_assets":   val_df['asset_id'].unique().tolist(),
    "test_assets":  test_df['asset_id'].unique().tolist(),
    "fitur_final":  FITUR_FINAL,
    "cutoff_train": str(cutoff_train),
    "cutoff_val":   str(cutoff_val),
    "n_train":      len(train_df),
    "n_val":        len(val_df),
    "n_test":       len(test_df),
}
with open(ARTIFACTS_DIR / "artifact_split.json", "w") as f:
    json.dump(split_meta, f, indent=2)
print("[OK] Split metadata saved.")

[ACTION] Capping RUL_hours at 5000h
[CENSORING] Failed (observed): 30 units | Censored (right): 0 units | Total retained: 30 units
[SPLIT] Train: 118,569 | Val: 14,816 | Test: 14,813
        Cutoff train: 2025-02-24 17:00:00 | Cutoff val: 2025-04-14 06:00:00
[OK] Scaler fitted on 118,569 train samples → saved

[LABEL DISTRIBUTION - Train]
  NORMAL    :  117,345 (99.0%)
  CRITICAL  :      408 (0.3%)
[OK] Split metadata saved.


In [5]:
# %%
# ==============================================================================
# DATA PIPELINE CELL 5 – LSTM Tensor Construction (per-unit, chronologically safe)
# ==============================================================================
HIERARCHY_TARGETS = [
    'RUL_hours',
    'RUL_hydraulic_system', 'RUL_hydraulic_pump', 'RUL_pump_seal_main',
    'RUL_brake_system',     'RUL_brake_caliper',  'RUL_brake_pad_rear',
    'RUL_steering_system',
]

def build_sequences_for_unit(
    unit_scaled: np.ndarray,
    unit_targets: dict,          # {col: np.ndarray}
    unit_gap_flag: np.ndarray,
    seq_len: int,
    unit_weight: float = 1.0,
) -> tuple:
    """
    Sliding window per unit. Lewati window yang mengandung gap.
    Menggunakan stride tricks untuk efisiensi — tidak ada Python loop per-window.
    """
    n, n_feat = unit_scaled.shape
    if n < seq_len:
        return None

    # Valid window indices (tidak mengandung gap)
    valid_starts = []
    for i in range(n - seq_len + 1):
        if not unit_gap_flag[i: i + seq_len].any():
            valid_starts.append(i)

    if not valid_starts:
        return None

    # Stride tricks untuk X
    shape   = (len(valid_starts), seq_len, n_feat)
    strides = (unit_scaled.strides[0], unit_scaled.strides[0], unit_scaled.strides[1])
    X_all   = np.lib.stride_tricks.as_strided(unit_scaled, shape=shape, strides=strides)
    X_seq   = X_all[np.array([i for i in range(len(valid_starts))], dtype=int)].copy()

    # Labels: ambil dari timestep terakhir window
    last_idx = np.array(valid_starts) + seq_len - 1
    y_dict   = {col: arr[last_idx] for col, arr in unit_targets.items()}
    weights  = np.full(len(valid_starts), unit_weight, dtype=np.float32)

    return X_seq, y_dict, weights


print(f"[TENSOR] Building LSTM sequences (window={SEQUENCE_LENGTH}h)...")
train_sorted = train_df.sort_values(['asset_id', 'timestamp']).reset_index(drop=True)

# Scale seluruh train_sorted dengan scaler yang sudah fitted
X_full_scaled = scaler.transform(
    train_sorted[FITUR_FINAL].values.astype(np.float32)
).astype(np.float32)

gap_arr = (train_sorted['connectivity_loss_flag'].values.astype(np.int8)
        if 'connectivity_loss_flag' in train_sorted.columns
        else np.zeros(len(train_sorted), dtype=np.int8))

all_X, all_y, all_w = [], {t: [] for t in HIERARCHY_TARGETS}, []
ptr = 0
stats = []

for asset_id, grp in train_sorted.groupby('asset_id', sort=True):
    n_unit   = len(grp)
    idx      = grp.index.values  # original index in train_sorted
    rel_idx  = np.arange(ptr, ptr + n_unit)

    unit_scaled  = X_full_scaled[rel_idx]
    unit_gap     = gap_arr[rel_idx]
    unit_targets = {
        col: grp[col].values.astype(np.float32)
        for col in HIERARCHY_TARGETS if col in grp.columns
    }
    unit_weight  = 1.0 if asset_id in failed_units else 0.5

    result = build_sequences_for_unit(
        unit_scaled, unit_targets, unit_gap, SEQUENCE_LENGTH, unit_weight
    )
    ptr += n_unit

    if result is None:
        stats.append(f"  [SKIP] {asset_id}: only {n_unit} rows < window {SEQUENCE_LENGTH}")
        continue

    X_seq, y_dict, w = result
    all_X.append(X_seq)
    for t in HIERARCHY_TARGETS:
        if t in y_dict:
            all_y[t].append(y_dict[t])
    all_w.append(w)

    rul_arr = grp['RUL_hours'].values
    stats.append(f"  {asset_id:>12}: {n_unit:>6} obs | "
                 f"RUL [{rul_arr.min():.0f}h–{rul_arr.max():.0f}h] | "
                 f"seq={len(X_seq):,} | weight={unit_weight}")

print("\n" + "="*65)
print("HEAVY EQUIPMENT UNIT LIST (Training)")
print("="*65)
for s in stats: print(s)
print("="*65)

assert len(all_X) > 0, "No valid sequences generated!"
X_train_lstm   = np.concatenate(all_X, axis=0)
y_train_reg    = np.concatenate(all_y['RUL_hours'], axis=0)
y_train_clf    = create_labels(y_train_reg)
sample_weights = np.concatenate(all_w, axis=0)
y_train_hier   = {t: np.concatenate(all_y[t]) for t in HIERARCHY_TARGETS if all_y[t]}

gap_total = 0  # already filtered out
print(f"\n[TENSOR] Shape: {X_train_lstm.shape} "
      f"(seq × timestep × feature)")
print(f"[TENSOR] NaN check: {np.isnan(X_train_lstm).sum()} NaN values")

print(f"\n[LABEL DIST] LSTM Training Sequences:")
uniq, cnts = np.unique(y_train_clf, return_counts=True)
for u, c in zip(uniq, cnts):
    print(f"  {LABEL_NAMES[u]:<10}: {c:,} ({c/len(y_train_clf)*100:.1f}%)")

# ── Save tensors ───────────────────────────────────────────────────────────────
np.save(ARTIFACTS_DIR / "X_train_lstm.npy",    X_train_lstm)
np.save(ARTIFACTS_DIR / "y_train_reg.npy",     y_train_reg)
np.save(ARTIFACTS_DIR / "y_train_clf.npy",     y_train_clf)
np.save(ARTIFACTS_DIR / "sample_weights.npy",  sample_weights)
print(f"\n[OK] Tensors saved to {ARTIFACTS_DIR}")

[TENSOR] Building LSTM sequences (window=24h)...

HEAVY EQUIPMENT UNIT LIST (Training)
      D155-001:   2848 obs | RUL [3944h–5000h] | seq=2,825 | weight=1.0
      D155-002:   3884 obs | RUL [0h–3883h] | seq=3,861 | weight=1.0
      D155-003:   3326 obs | RUL [0h–3325h] | seq=3,303 | weight=1.0
      D155-004:   6003 obs | RUL [0h–5000h] | seq=5,980 | weight=1.0
      D155-005:   1595 obs | RUL [1244h–2838h] | seq=1,572 | weight=1.0
      D155-006:   2275 obs | RUL [2453h–4727h] | seq=2,252 | weight=1.0
     HD785-001:   2535 obs | RUL [0h–2534h] | seq=2,512 | weight=1.0
     HD785-002:   5928 obs | RUL [0h–5000h] | seq=5,905 | weight=1.0
     HD785-003:   2577 obs | RUL [2021h–4597h] | seq=2,554 | weight=1.0
     HD785-004:   2515 obs | RUL [0h–2514h] | seq=2,492 | weight=1.0
     HD785-005:   3209 obs | RUL [0h–3208h] | seq=3,186 | weight=1.0
     HD785-006:   1552 obs | RUL [3607h–5000h] | seq=1,529 | weight=1.0
     HD785-007:   3431 obs | RUL [2984h–5000h] | seq=3,408 | weight=1.

In [6]:
# %%
# ==============================================================================
# DATA PIPELINE CELL 6 – Export Pilot Dataset (subset untuk demo/testing)
# ==============================================================================
PILOT_ASSETS  = ['HD785-001', 'HD785-002', 'PC2000-001', 'D155-001', 'WA600-001']
PILOT_HOURS   = 500  # ambil 500 jam terakhir per unit

pilot_frames = []
for asset in PILOT_ASSETS:
    sub = df[df['asset_id'] == asset].sort_values('timestamp').tail(PILOT_HOURS)
    if len(sub) > 0:
        pilot_frames.append(sub)
    else:
        print(f"[WARN] {asset} not found in dataset")

if pilot_frames:
    df_pilot = pd.concat(pilot_frames, ignore_index=True)
    pilot_path = DATA_DIR / "dataset_pratyaksa_pilot.parquet"
    df_pilot.to_parquet(pilot_path, index=False)
    print(f"[OK] Pilot dataset → {pilot_path}")
    print(f"     {len(df_pilot)} rows | {df_pilot['asset_id'].nunique()} units")
    print(f"     RUL range: {df_pilot['RUL_hours'].min():.0f}h – {df_pilot['RUL_hours'].max():.0f}h")

print("\n[PIPELINE COMPLETE] All artifacts ready:")
for f in sorted(ARTIFACTS_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f"  {f.name:<40} {size:>8.1f} KB")

[OK] Pilot dataset → /home/sleepy/projects/project-pratyaksa-kic/data/dataset_pratyaksa_pilot.parquet
     2500 rows | 5 units
     RUL range: 0h – 499h

[PIPELINE COMPLETE] All artifacts ready:
  X_train_lstm.npy                         408892.9 KB
  _ckpt_bulldozer.keras                      1819.9 KB
  _ckpt_excavator.keras                      1819.9 KB
  _ckpt_haul_truck.keras                     1819.9 KB
  _ckpt_wheel_loader.keras                   1819.9 KB
  artifact_deploy_meta.json                     4.1 KB
  artifact_lstm_bulldozer.keras              1819.9 KB
  artifact_lstm_excavator.keras              1819.9 KB
  artifact_lstm_haul_truck.keras             1819.9 KB
  artifact_lstm_model.keras                  1807.3 KB
  artifact_lstm_wheel_loader.keras           1819.9 KB
  artifact_scaler.pkl                           1.4 KB
  artifact_split.json                           2.1 KB
  artifact_xgb_model.json                    1001.8 KB
  artifact_xgb_model.onnx          